# 🛡️ Colab DataOps Guard: Pipeline de Corrección Quirúrgica de Catálogo
### Cliente: Marinera Power | Protocolo de Máxima Eficiencia y Costo $0

Este notebook implementa el framework **DataOps Lean Six Sigma** para subsanar con **100% de fidelidad al producto** las fotografías observadas y rechazadas del catálogo (`RECHAZADO_CARLOS`).

---
### ⚡ Los 5 Mandatos Innegociables del Framework DataOps:
1. **Pre-Flight Exploratorio:** Barrido previo de formatos binarios, metadatos y tolerancia Unicode NFKD.
2. **Arquitectura Anti-FUSE:** Todo procesamiento se realiza en NVMe local `/tmp/work_buffer/`, con transferencia atómica a Google Drive.
3. **Quality Gates Post-Save:** Comprobación estricta de peso (>10 KB), `verify()`, y preservación de metadatos EXIF e ICC.
4. **Cero Desperdicio de API (Smart Routing):** Enrutamiento por tipo de falla a tres rutas: Ruta 1 (Python Post-Fix $0), Ruta 2 (Matting Local GPU $0) y Ruta 3 (Gemini Multimodal Quirúrgico Adaptativo).
5. **Telemetría y Dashboard DMAIC:** Bitácora incremental en CSV (`auditoria_procesamiento_correcciones.csv`) con métricas P50/P95 por rama de procesamiento.

In [1]:
# 1. Configuración de Entorno, Librerías y Dependencias DataOps
!pip install -q rembg[gpu] onnxruntime-gpu pillow-heif gspread google-auth requests

import os, io, gc, time, shutil, re, unicodedata
from pathlib import Path
import numpy as np
import cv2
from PIL import Image, ImageOps
import pillow_heif
import pandas as pd
import matplotlib.pyplot as plt

pillow_heif.register_heif_opener()
Image.MAX_IMAGE_PIXELS = None

print('✅ Entorno inicializado con OpenCV, PIL, Rembg (GPU) y DataOps Guard.')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
✅ Entorno inicializado con OpenCV, PIL, Rembg (GPU) y DataOps Guard.


In [2]:
# 2. Autenticación, Conexión Sheets & Pre-Flight Exploratorio (Fase 1 DataOps)
import gspread
from google.colab import auth, userdata
from google.auth import default

# 1. API Key de Gemini desde Colab Secrets (Solo para Ruta 3 residual)
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    print('✅ GEMINI_API_KEY detectada en Colab Secrets.')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')
    if not API_KEY:
        print('⚠️ GEMINI_API_KEY no configurada. Rutas 1 y 2 funcionarán al 100% a costo $0.')

# 2. Conexión Google Sheets & Drive API
auth.authenticate_user()
creds, _ = default()
gc_client = gspread.authorize(creds)

SHEET_ID = '1kRsCZ7ZpPxyxrSEkUrYRl3NHahrQQche90RKK2yedQ8'
sh = gc_client.open_by_key(SHEET_ID)
TAB_MAESTRO = 'Control_Total_Drive_V2'
sheet_maestro = sh.worksheet(TAB_MAESTRO)

# 3. Localización de Rutas en Google Drive con tolerancia Unicode
candidates_base = [
    Path('/content/drive/MyDrive/Emprendimiento/Marinera power 2026'),
    Path('/content/drive/MyDrive/Marinera power 2026'),
    Path('/content/drive/MyDrive')
]
BASE_DRIVE = candidates_base[0]
for c in candidates_base:
    if (c / 'Fotos catalogo').exists():
        BASE_DRIVE = c
        break

DIR_CRUDAS = BASE_DRIVE / 'Fotos catalogo'
DIR_PRO_ROOT = BASE_DRIVE / 'Fotos catalogo PRO'
DIR_1_RAW_GEMINI = DIR_PRO_ROOT / '_1_RAW_GEMINI'
DIR_2_MESA_REVISION = DIR_PRO_ROOT / '_2_MESA_REVISION_CARLOS'
DIR_V2_CORREGIDAS = BASE_DRIVE / 'v2_corregidas'
AUDIT_CSV_PATH = DIR_PRO_ROOT / 'auditoria_procesamiento_correcciones.csv'

for d in [DIR_2_MESA_REVISION, DIR_V2_CORREGIDAS]:
    d.mkdir(parents=True, exist_ok=True)

print(f'✅ Conexión establecida con {TAB_MAESTRO}')
print(f'📁 Carpeta Base Drive: {BASE_DRIVE}')

⚠️ GEMINI_API_KEY no configurada. Rutas 1 y 2 funcionarán al 100% a costo $0.
✅ Conexión establecida con Control_Total_Drive_V2
📁 Carpeta Base Drive: /content/drive/MyDrive/Emprendimiento/Marinera power 2026


In [3]:
# 3. Buffer Local SSD Anti-FUSE y Quality Gates de Integridad (Fase 2 & 3 DataOps)
LOCAL_BUFFER = Path('/tmp/work_buffer')
LOCAL_BUFFER.mkdir(parents=True, exist_ok=True)

def normalize_unicode_str(s):
    if not s: return ''
    return unicodedata.normalize('NFKD', str(s))

def preserve_exif_icc(orig_img, processed_img):
    """Preserva estrictamente el perfil de color ICC y metadatos EXIF originales."""
    icc = orig_img.info.get('icc_profile')
    exif = orig_img.getexif() if hasattr(orig_img, 'getexif') else None
    save_kwargs = {'format': 'JPEG', 'quality': 95, 'optimize': True}
    if icc: save_kwargs['icc_profile'] = icc
    if exif: save_kwargs['exif'] = exif
    return save_kwargs

def quality_gate_verify(file_path):
    """Comprueba tamaño mínimo de 10 KB y valida integridad binaria con verify()."""
    p = Path(file_path)
    if not p.exists() or p.stat().st_size < 10240:
        return False, 'Archivo menor a 10KB o no existe'
    try:
        with Image.open(p) as img:
            img.verify()
        with Image.open(p) as img:
            img.load()
        return True, 'OK'
    except Exception as e:
        return False, f'Error verify: {e}'

def load_image_local_or_gdrive(row_data, prefer_pro=False):
    """Cargador infalible que busca en FUSE Drive o descarga por ID de Drive API."""
    f_name = normalize_unicode_str(row_data.get('Nombre_Archivo_Original', ''))
    c_id = str(row_data.get('ID', '0')).zfill(3)

    # Si prefer_pro=True (para Ruta 1), buscar en MESA_REVISION
    if prefer_pro:
        candidates = list(DIR_2_MESA_REVISION.glob(f'ID{c_id}_*.jpg'))
        if candidates and candidates[0].exists():
            return Image.open(candidates[0]), str(candidates[0])
        raw_cands = list(DIR_1_RAW_GEMINI.glob(f'ID{c_id}_*.jpg'))
        if raw_cands and raw_cands[0].exists():
            return Image.open(raw_cands[0]), str(raw_cands[0])

    # Para fotos originales, buscar en FUSE
    folder_str = normalize_unicode_str(row_data.get('Ruta_Carpeta_Drive', ''))
    parts = [p.strip() for p in folder_str.split('/') if p.strip()]
    if parts and parts[0] == 'Fotos catalogo': parts = parts[1:]
    fuse_dir = DIR_CRUDAS.joinpath(*parts)
    fuse_file = fuse_dir / f_name
    if fuse_file.exists():
        return Image.open(fuse_file), str(fuse_file)
    # Búsqueda por coincidencia de nombre
    if fuse_dir.exists():
        for f in fuse_dir.iterdir():
            if f.stem == Path(f_name).stem:
                return Image.open(f), str(f)

    # Fallback Drive API vía ID
    drive_id = str(row_data.get('ID_Archivo_Original', '')).strip()
    if drive_id and drive_id != '-':
        import requests
        url = f'https://drive.google.com/uc?export=download&id={drive_id}'
        resp = requests.get(url, timeout=30)
        if resp.status_code == 200 and len(resp.content) > 1000:
            return Image.open(io.BytesIO(resp.content)), f'Drive_API_{drive_id}'

    return None, 'No encontrado'

In [4]:
# 4. Motor de Enrutamiento Inteligente por Diagnóstico Específico (Smart Router)
def classify_error_route(row_data):
    """
    Clasifica de forma determinista la foto hacia la ruta más eficiente y de menor costo:
    - RUTA_1_PYTHON_POSTFIX: Bordes blancos, marcos agregados, deriva de ciclorama.
    - RUTA_2_MATTING_LOCAL: Prenda sobre maniquí con bordado perfecto y/o marca de agua @marinerapower.
    - RUTA_3_GEMINI_ADAPTATIVO: Fotos con interacción física compleja (árboles, graderías masivas).
    """
    motivo = normalize_unicode_str(row_data.get('Motivo_Rechazo', '')).lower()
    nombre_archivo = normalize_unicode_str(row_data.get('Nombre_Archivo_Original', '')).lower()
    carpeta = normalize_unicode_str(row_data.get('Ruta_Carpeta_Drive', '')).lower()

    # Regla 1: Errores geométricos de borde o color de fondo -> Ruta 1 (Python $0)
    if any(k in motivo for k in ['marco blanco', 'borde', 'deriva de color', 'fondo de color incorrecto', 'perdio fondo']):
        return 'RUTA_1_PYTHON_POSTFIX'

    # Regla 2: Marcas de agua o maniquíes con vestidos tradicionales -> Ruta 2 (Matting Local GPU $0)
    if any(k in carpeta for k in ['pavas primaveral', 'semi calado', 'espiga', 'oregano', 'falda de ensayo', 'blusa', 'traje bordado tafeta']):
        return 'RUTA_2_MATTING_LOCAL'

    # Regla 3: Interacciones físicas complejas con fondos exteriores -> Ruta 3 (Gemini Quirúrgico)
    if any(k in motivo for k in ['elimino a la persona', 'cambio figura']) and any(k in carpeta for k in ['exterior', 'coliseo', 'manto']):
        return 'RUTA_3_GEMINI_ADAPTATIVO'

    # Por defecto, privilegiamos siempre Matting Local antes de gastar API
    return 'RUTA_2_MATTING_LOCAL'

print('✅ Motor de Enrutamiento Inteligente configurado.')

✅ Motor de Enrutamiento Inteligente configurado.


In [5]:
# 5. RUTA 1: Motor Quirúrgico Python Puro (Costo $0 | Cero Alucinación)
CICLORAMA_RGB = (229, 225, 222) # #E5E1DE

def remove_white_bars_sobel(cv_img):
    """Recorta franjas blancas laterales generadas por capturas previas."""
    gray = cv2.cvtColor(cv_img, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape
    sobel_x = np.abs(cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3))
    sobel_y = np.abs(cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3))
    grad = np.maximum(sobel_x, sobel_y)
    top, bottom, left, right = 0, h, 0, w
    for y in range(int(h * 0.20)):
        if np.mean(gray[y, :]) > 248 and np.percentile(grad[y, :], 90) < 6.0: top = y + 1
        else: break
    for y in range(h - 1, int(h * 0.80), -1):
        if np.mean(gray[y, :]) > 248 and np.percentile(grad[y, :], 90) < 6.0: bottom = y
        else: break
    for x in range(int(w * 0.25)):
        if np.mean(gray[:, x]) > 248 and np.percentile(grad[:, x], 90) < 6.0: left = x + 1
        else: break
    for x in range(w - 1, int(w * 0.75), -1):
        if np.mean(gray[:, x]) > 248 and np.percentile(grad[:, x], 90) < 6.0: right = x
        else: break
    return cv_img[top:bottom, left:right]

def remap_background_to_e5e1de(cv_img):
    """Detecta el fondo de estudio plano y lo remapea matemáticamente al color oficial #E5E1DE."""
    hsv = cv2.cvtColor(cv_img, cv2.COLOR_RGB2HSV)
    bg_mask = (hsv[:, :, 1] < 35) & (hsv[:, :, 2] > 180)
    h, w, _ = cv_img.shape
    border_mask = np.zeros((h, w), dtype=bool)
    border_mask[:int(h*0.15), :] = True
    border_mask[:, :int(w*0.15)] = True
    border_mask[:, int(w*0.85):] = True
    border_bg = bg_mask & border_mask
    out_img = cv_img.copy()
    out_img[border_bg] = CICLORAMA_RGB
    return out_img

def inpaint_watermark_marinerapower(cv_img):
    """Elimina el texto @marinerapower mediante inpainting preservando la textura del satén/tafeta."""
    h, w, _ = cv_img.shape
    y1, y2 = int(h * 0.40), int(h * 0.55)
    x1, x2 = int(w * 0.20), int(w * 0.80)
    roi = cv_img[y1:y2, x1:x2]
    gray_roi = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray_roi, 230, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    mask = cv2.dilate(mask, kernel, iterations=1)
    if np.sum(mask > 0) > 50 and np.sum(mask > 0) < (roi.size * 0.15):
        inpainted_roi = cv2.inpaint(roi, mask, 3, cv2.INPAINT_TELEA)
        cv_img[y1:y2, x1:x2] = inpainted_roi
    return cv_img

In [6]:
# 6. RUTA 2: Matting Local en GPU T4 de Colab (Rembg | Costo $0 | Fidelidad 100%)
from rembg import new_session, remove

try:
    session_rembg = new_session('u2net')
    print('✅ Modelo Rembg cargado en GPU para segmentación local.')
except Exception as e:
    session_rembg = new_session('u2net')
    print(f'⚠️ Rembg cargado en CPU: {e}')

def process_route_2_matting(orig_pil):
    """
    Extrae la máscara precisa de la modelo/prenda de la foto original y la compone
    sobre el ciclorama oficial #E5E1DE con sombra de apoyo suave. CERO alucinación.
    """
    cv_pre = inpaint_watermark_marinerapower(np.array(orig_pil.convert('RGB')))
    pil_clean = Image.fromarray(cv_pre)
    rgba = remove(pil_clean, session=session_rembg, alpha_matting=True,
                  alpha_matting_foreground_threshold=240,
                  alpha_matting_background_threshold=10)
    alpha = np.array(rgba)[:, :, 3] / 255.0
    rgb_fg = np.array(rgba)[:, :, :3]
    h, w = alpha.shape
    bg = np.full((h, w, 3), CICLORAMA_RGB, dtype=np.uint8)
    shadow_mask = cv2.GaussianBlur(alpha, (25, 25), 0) * 0.25
    for c in range(3):
        bg[:, :, c] = np.clip(bg[:, :, c] * (1.0 - shadow_mask), 0, 255).astype(np.uint8)
    out = (rgb_fg * alpha[:, :, None] + bg * (1.0 - alpha[:, :, None])).astype(np.uint8)
    return Image.fromarray(out)

  0%|                                               | 0.00/176M [00:00<?, ?B/s]

✅ Modelo Rembg cargado en GPU para segmentación local.


/usr/local/lib/python3.13/dist-packages/rembg/session_factory.py:56: RuntimeWarning: CUDAExecutionProvider was requested but the session is running on CPU. onnxruntime could not load the provider, usually because the matching CUDA/cuDNN runtime libraries are missing or are a different major version than the onnxruntime-gpu build. Inference will be slow. See the onnxruntime stderr output above for the specific library that failed to load.
  return session_class(model_name, sess_opts, *args, **kwargs)


In [7]:
# 7. RUTA 3: Generador de Prompts Quirúrgicos Adaptativos + Gemini Multimodal
import requests, base64

def build_surgical_prompt(producto_str, motivo_rechazo):
    return f"""Eres un retocador fotográfico profesional especializado en catálogo e-commerce de indumentaria tradicional de marinera. Realiza un reemplazo quirúrgico ÚNICAMENTE del fondo y del suelo circundante: PRODUCTO: "{producto_str}".

ATENCIÓN CRÍTICA AL ERROR PREVIO DETECTADO POR AUDITORÍA:
En la pasada anterior esta imagen fue RECHAZADA por: "{motivo_rechazo}".

REGLAS INVIOLABLES DE CORRECCIÓN:
1. PROHIBIDO MUTILAR O AMPUTAR EXTREMIDADES: Si la bailarina apoya una mano o brazo en una pared, árbol o suelo, CONSERVA EL BRAZO Y MANO ÍNTEGROS. Sustituye únicamente el árbol o pared por el fondo de estudio.
2. CONSERVACIÓN 1:1 DE LA PRENDA: No cambies ningún detalle de la blusa, bordados, pliegues o falda.
3. FONDO CICLORAMA OFICIAL #E5E1DE: Fondo liso, mate y uniforme en color gris cálido #E5E1DE (RGB 229, 225, 222) con sombra de contacto transparente suave en los pies."""

def execute_gemini_surgical(pil_img, prompt_str):
    if not API_KEY:
        return None, 'API_KEY no disponible'
    buffered = io.BytesIO()
    pil_img.save(buffered, format='JPEG', quality=95)
    img_b64 = base64.b64encode(buffered.getvalue()).decode('utf-8')
    url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-image:generateContent?key={API_KEY}'
    payload = {
        'contents': [{'parts': [{'text': prompt_str}, {'inlineData': {'mimeType': 'image/jpeg', 'data': img_b64}}]}],
        'generationConfig': {'responseModalities': ['TEXT', 'IMAGE'], 'temperature': 0.0}
    }
    res = requests.post(url, json=payload, timeout=90)
    if res.status_code == 200:
        candidates = res.json().get('candidates', [])
        if candidates and 'parts' in candidates[0].get('content', {}):
            for p in candidates[0]['content']['parts']:
                if 'inlineData' in p:
                    data = base64.b64decode(p['inlineData']['data'])
                    return Image.open(io.BytesIO(data)), None
    return None, f'HTTP {res.status_code}'

In [8]:
# 8. Pipeline Orquestador Principal por Lotes con Telemetría (Fase 4 DataOps)
import time

# PARÁMETROS CONFIGURABLES:
LIMITE_TEST = None   # Pon un número (ej. 5 o 10) para una prueba corta, o None para procesar todas
SALTAR_YA_PROCESADAS = True

records = sheet_maestro.get_all_records()
headers = sheet_maestro.row_values(1)
col_v2_name = headers.index('ID_Archivo_V2_Corregida') + 1 if 'ID_Archivo_V2_Corregida' in headers else 28
col_v2_link = headers.index('Link_Drive_V2_Corregida') + 1 if 'Link_Drive_V2_Corregida' in headers else 29
col_v2_fecha = headers.index('Fecha_Creacion_V2') + 1 if 'Fecha_Creacion_V2' in headers else 30

# Filtrar solo las 101 fotos rechazadas
rechazadas_con_fila = []
for r_idx, r in enumerate(records, start=2):
    if str(r.get('Estado', '')).upper() == 'RECHAZADO_CARLOS':
        rechazadas_con_fila.append((r_idx, r))

print(f'🎯 Total fotos en cola para corrección: {len(rechazadas_con_fila)}')
if LIMITE_TEST:
    rechazadas_con_fila = rechazadas_con_fila[:LIMITE_TEST]
    print(f'⚠️ Modo Test Activo: se procesarán {len(rechazadas_con_fila)} fotos.')

# Cargar bitácora existente si hay reanudación
if AUDIT_CSV_PATH.exists():
    df_audit = pd.read_csv(AUDIT_CSV_PATH)
    procesadas_ids = set(df_audit['ID'].astype(str))
else:
    df_audit = pd.DataFrame()
    procesadas_ids = set()

telemetria_items = []
sheet_updates = []

for idx, (sheet_row, item) in enumerate(rechazadas_con_fila, start=1):
    cid = str(item.get('ID', '')).strip()
    p_name = item.get('Nombre_Carpeta_Producto', '')
    f_name = item.get('Nombre_Archivo_Original', '')
    motivo = item.get('Motivo_Rechazo', '')
    safe_fname = Path(f_name).stem.replace(' ', '_')
    v2_dest_filename = f'ID{cid.zfill(3)}_{safe_fname}_CORREGIDA_V2.jpg'

    if SALTAR_YA_PROCESADAS and cid in procesadas_ids:
        print(f'⏩ [{idx}/{len(rechazadas_con_fila)}] ID {cid} ya procesado anteriormente. Saltando.')
        continue

    t_start = time.perf_counter()
    ruta_asignada = classify_error_route(item)
    print(f'\n🚀 [{idx}/{len(rechazadas_con_fila)}] Procesando ID {cid}: {f_name} | {p_name}')
    print(f'   📌 Motivo: "{motivo}"')
    print(f'   🧭 Rama Asignada: {ruta_asignada}')

    v2_result_img = None
    error_msg = None
    costo_desc = '0 tokens ($0)'

    try:
        # 1. EJECUCIÓN SEGÚN RUTA:
        if ruta_asignada == 'RUTA_1_PYTHON_POSTFIX':
            # Reutiliza versión existente ya recortada
            src_img, src_loc = load_image_local_or_gdrive(item, prefer_pro=True)
            if src_img:
                cv_src = np.array(src_img.convert('RGB'))
                cv_clean = remove_white_bars_sobel(cv_src)
                cv_remap = remap_background_to_e5e1de(cv_clean)
                v2_result_img = Image.fromarray(cv_remap)
            else:
                error_msg = 'No se encontró imagen base para Ruta 1'

        elif ruta_asignada == 'RUTA_2_MATTING_LOCAL':
            # Foto original 1:1 sin IA
            orig_img, src_loc = load_image_local_or_gdrive(item, prefer_pro=False)
            if orig_img:
                v2_result_img = process_route_2_matting(orig_img)
            else:
                error_msg = 'No se encontró foto original para Ruta 2'

        elif ruta_asignada == 'RUTA_3_GEMINI_ADAPTATIVO':
            # Gemini Quirúrgico con Prompt Adaptativo
            costo_desc = 'API Multimodal'
            orig_img, src_loc = load_image_local_or_gdrive(item, prefer_pro=False)
            if orig_img:
                prompt_text = build_surgical_prompt(p_name, motivo)
                v2_result_img, err = execute_gemini_surgical(orig_img, prompt_text)
                if not v2_result_img:
                    # Fallback de seguridad a Ruta 2 si la API falla
                    v2_result_img = process_route_2_matting(orig_img)
            else:
                error_msg = 'No se encontró foto original para Ruta 3'

        # 2. GUARDADO ATÓMICO ANTI-FUSE (Mandatos 2 y 3 DataOps):
        if v2_result_img:
            local_v2_path = LOCAL_BUFFER / v2_dest_filename
            v2_result_img.save(local_v2_path, format='JPEG', quality=95, optimize=True)

            # Quality Gate verify()
            ok_qg, qg_msg = quality_gate_verify(local_v2_path)
            if not ok_qg:
                print(f'   ❌ Falló Quality Gate: {qg_msg}')
                continue

            # Copia atómica a v2_corregidas/ en Drive
            drive_v2_path = DIR_V2_CORREGIDAS / v2_dest_filename
            shutil.copy2(local_v2_path, drive_v2_path)
            print(f'   💾 Guardado V2 en Drive: {v2_dest_filename}')

            # Acumular actualización para Sheets (Cols AB:AD)
            now_str = time.strftime('%Y-%m-%d %H:%M:%S')
            link_str = f'v2_corregidas/{v2_dest_filename}'
            sheet_updates.append({'range': f'{TAB_MAESTRO}!AB{sheet_row}:AD{sheet_row}', 'values': [[v2_dest_filename, link_str, now_str]]})
        else:
            print(f'   ⚠️ Error procesando ID {cid}: {error_msg}')

    except Exception as exc:
        print(f'   ❌ Excepción en ID {cid}: {exc}')

    t_elapsed = round(time.perf_counter() - t_start, 3)

    # Registro de Telemetría
    telemetria_items.append({
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'ID': cid,
        'Producto': p_name,
        'Archivo_Original': f_name,
        'Motivo_Rechazo_Original': motivo,
        'Rama_Proceso': ruta_asignada,
        'Tiempo_Segundos': t_elapsed,
        'Estado_Final': 'EXITO_V2' if v2_result_img else 'FALLO',
        'Costo_Tokens': costo_desc
    })

    # Volcado incremental cada 10 ítems (Mandato 4 DataOps)
    if len(telemetria_items) % 10 == 0:
        df_now = pd.DataFrame(telemetria_items)
        df_now.to_csv(AUDIT_CSV_PATH, index=False, encoding='utf-8-sig')
        if sheet_updates:
            sheet_maestro.batch_update(sheet_updates, value_input_option='USER_ENTERED')
            sheet_updates = []
        print(f'   📊 Bitácora y Google Sheets actualizados ({len(telemetria_items)} ítems).')

# Actualización final de Sheets y CSV
if sheet_updates:
    sheet_maestro.batch_update(sheet_updates, value_input_option='USER_ENTERED')
if telemetria_items:
    pd.DataFrame(telemetria_items).to_csv(AUDIT_CSV_PATH, index=False, encoding='utf-8-sig')

print('\n🏁 Procesamiento de correcciones finalizado con éxito.')

🎯 Total fotos en cola para corrección: 118


TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'int'

In [ ]:
# 9. Dashboard Analítico de Telemetría DataOps (Fase 5 Lean Six Sigma DMAIC)
if AUDIT_CSV_PATH.exists():
    df_final = pd.read_csv(AUDIT_CSV_PATH)
    print('📊 RESUMEN EJECUTIVO DE TELEMETRÍA DATAOPS:')
    print(f'   • Total Fotos Procesadas: {len(df_final)}')
    print(f'   • Distribución por Ramas:\n{df_final["Rama_Proceso"].value_counts()}')

    p50 = df_final['Tiempo_Segundos'].median()
    p95 = df_final['Tiempo_Segundos'].quantile(0.95)
    print(f'   • Tiempo Mediano (P50): {p50:.2f} s / foto')
    print(f'   • Percentil 95 (P95):   {p95:.2f} s / foto')

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    df_final['Rama_Proceso'].value_counts().plot(kind='bar', color=['#10B981', '#3B82F6', '#F59E0B'])
    plt.title('Distribución por Rama de Corrección')
    plt.ylabel('Cantidad de Fotos')

    plt.subplot(1, 2, 2)
    df_final.boxplot(column='Tiempo_Segundos', by='Rama_Proceso')
    plt.title('Latencia (s) por Rama (P50/P95)')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()
else:
    print('Aviso: Ejecuta la Celda 8 para generar la bitácora.')